# SVM v3 — Improved Training + Calibration

Changes: expanded training data (all sessions from non-interday train subjects), calibration combines base + target features.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

from config import RANDOM_SEED, MODELS_DIR
from src.experiment_runner import (
    get_splits, load_and_norm, split_cal_test,
    run_zero_shot, run_calibration, print_comparison,
    TEST_SUBJECTS, TRAIN_SUBJECTS,
)
from src.feature_extraction import extract_features_batch
from src.evaluation import measure_latency, print_latency

splits = get_splits()
print(f'Train subjects: {TRAIN_SUBJECTS}')
print(f'Test subjects:  {TEST_SUBJECTS}')

Train subjects: ['h0', 'h1', 'h10', 'h11', 'h12', 'h13', 'h14', 'h15', 'h18', 'h19', 'h2', 'h20', 'h21', 'h23', 'h25', 'h26', 'h27', 'h28', 'h29', 'h4', 'h5', 'h6', 'h8', 'h9']
Test subjects:  ['h7', 'h22', 'h3', 'h24', 'h16', 'h17']


In [2]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

print('Extracting features...')
F_train_all = extract_features_batch(X_train)
y_train_all = y_train
print(f'Features: {F_train_all.shape}')

MAX_SVM = 50000
idx = resample(np.arange(len(y_train_all)), n_samples=MAX_SVM,
               random_state=RANDOM_SEED, stratify=y_train_all)
F_base, y_base = F_train_all[idx], y_train_all[idx]
print(f'Subsampled to {MAX_SVM}')

scaler = StandardScaler().fit(F_base)
F_base_scaled = scaler.transform(F_base)

print('Training SVM...')
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_SEED)
svm.fit(F_base_scaled, y_base)
print('Done.')

Loading windows: 100%|██████████| 9021/9021 [00:05<00:00, 1641.80it/s]


Train: (1030712, 8, 50)
Extracting features...
Features: (1030712, 120)
Subsampled to 50000
Training SVM...
Done.


In [3]:
def svm_predict(X):
    return svm.predict(scaler.transform(extract_features_batch(X)))

def svm_finetune(X_cal, y_cal):
    # Combine: 5000 base samples + all calibration samples (upsampled 3x)
    F_cal = extract_features_batch(X_cal)
    n_base = min(5000, len(F_base))
    idx_b = resample(np.arange(len(y_base)), n_samples=n_base,
                     random_state=RANDOM_SEED, stratify=y_base)
    # Upsample calibration data to balance with base
    n_repeat = max(1, n_base // (len(F_cal) * 2))
    F_combined = np.vstack([F_base[idx_b]] + [F_cal] * n_repeat)
    y_combined = np.concatenate([y_base[idx_b]] + [y_cal] * n_repeat)
    F_combined = scaler.transform(F_combined)
    svm_ft = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_SEED)
    svm_ft.fit(F_combined, y_combined)
    def predict_ft(X):
        return svm_ft.predict(scaler.transform(extract_features_batch(X)))
    return predict_ft

print('Option B — Zero-shot:')
zero_results = run_zero_shot(svm_predict, splits, norm_stats)
print('\nOption A — Calibration:')
cal_results = run_calibration(svm_predict, svm_finetune, splits, norm_stats)

Option B — Zero-shot:
  S1 zero-shot: 0.4624
  S2 zero-shot: 0.3915
  S3 zero-shot: 0.3979
  S4 zero-shot: 0.4759
  S5 zero-shot: 0.5326

Option A — Calibration:
  S1 calibrated: 0.5725
  S2 calibrated: 0.5491
  S3 calibrated: 0.5621
  S4 calibrated: 0.5507
  S5 calibrated: 0.6493


In [4]:
sample = X_train[:1]
def svm_single(x): return svm.predict(scaler.transform(extract_features_batch(x)))
latency = measure_latency(svm_single, sample, n_runs=500)
print_latency(latency, 'SVM')
print_comparison(zero_results, cal_results, name='SVM (v3)')


Latency — SVM
  Mean:   4.36 ms
  Median: 4.27 ms
  P95:    4.75 ms
  <300ms: ✓

  SVM (v3) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                46.24%       57.25%   +11.01%
S2                39.15%       54.91%   +15.77%
S3                39.79%       56.21%   +16.43%
S4                47.59%       55.07%    +7.48%
S5                53.26%       64.93%   +11.67%
